# Example Usage of racer<sup>TS<sup>

### Imports

In [ ]:
from racerts import ConformerGenerator
from rdkit.Chem.Draw import IPythonConsole
from rdkit import Chem

### Example Transition State: Cyclization

In [ ]:
file_name = 'ts.xyz'
charge = 0
reacting_atoms = [7,8,22]
input_smiles = ["C/[NH+]=C(OC)/c1ccccc1.COS(=O)(=O)[O-]"]

In [ ]:
cg = ConformerGenerator()

In [ ]:
mol = cg.get_mol(file_name, charge, reacting_atoms, input_smiles)
IPythonConsole.ipython_3d = True
mol

In [ ]:
IPythonConsole.ipython_3d = False
mol.RemoveAllConformers()
mol.__sssAtoms = reacting_atoms
mol.__sssQry = None
mol

### Transition state conformer generation

In [ ]:
ts_conformers_mol = cg.generate_conformers(file_name=file_name, charge = charge, reacting_atoms=reacting_atoms, input_smiles=input_smiles)
ts_conformers_mol.GetNumConformers()

In [ ]:
from racerts.visualizer import drawit
drawit(ts_conformers_mol)

### Exploring Options

In [ ]:
from racerts.mol_getter import MolGetterBonds, MolGetterConnectivity, MolGetterSMILES

cg.mol_getter = MolGetterBonds(
    assignBonds = True, 
    allowChargedFragments = True,
)
cg.mol_getter = MolGetterConnectivity()
cg.mol_getter = MolGetterSMILES()


In [ ]:
from racerts.embedder import BoundsMatrixEmbedder, CmapEmbedder

cg.embedder = BoundsMatrixEmbedder(
    verbose = False,
    randomSeed = 12,
    remove_all_conformers = True,
    ETversion = 2,
    useRandomCoords = True,
)
cg.embedder = CmapEmbedder(
    verbose = False,
    randomSeed = 12,
    pruneRmsThresh = -1,
    remove_all_conformers = True,
    ETversion = 2,
    useRandomCoords = True,
)

In [ ]:
from racerts.optimizer import MMFFOptimizer, UFFOptimizer

cg.optimizer = MMFFOptimizer(
    verbose = False,
    conf_id_ref= -1,
    force_constant=1000000,
)
cg.optimizer = UFFOptimizer(
    verbose = False,
    conf_id_ref= -1,
    force_constant=1000000,
)

In [ ]:
from racerts.pruner import EnergyPruner, RMSDPruner

cg.energy_pruner = EnergyPruner(
    threshold = 6.0, 
    verbose = False,
)

cg.rmsd_pruner = RMSDPruner(
    threshold=0.125,
    verbose=False
)

In [ ]:
ts_conformers_mol = cg.get_mol(file_name=file_name, charge = charge, reacting_atoms=reacting_atoms)
ts_conformers_mol

In [ ]:
cg.embedder.pruneRmsThresh = -1
ts_conformers_mol = cg.generate_conformers(file_name=file_name, charge = charge, reacting_atoms=reacting_atoms)
ts_conformers_mol.GetNumConformers()

### ASE calculators

In [ ]:
!pip install ase

In [ ]:
from racerts.optimizer import ASEOptimizer
from ase.constraints import FixAtoms
from tblite.ase import TBLite # requires tblite to be installed, see https://tblite.readthedocs.io/en/latest/users/ase.html

In [ ]:
m = Chem.Mol(ts_conformers_mol)
optimizer = ASEOptimizer(TBLite(verbosity=0))
constraints = FixAtoms(indices=reacting_atoms)

optimizer.optimize(m, constraints=constraints, conf_id=-1)

In [ ]:
gen = ConformerGenerator()
gen.optimizer = ASEOptimizer(TBLite(verbosity=1))
ts_mol = gen.generate_conformers(file_name=file_name, charge=0, reacting_atoms=reacting_atoms, input_smiles=input_smiles)